# 05 - Object detection (all videos, optimized)

Process every video represented by a `keyframes.parquet` under `/kaggle/input`. The fixed-vocabulary YOLO26 model is loaded once and reused for the complete batch. Each video gets its own output directory, success marker, and error log.

The production default processes every keyframe. Set `MAX_VIDEOS` or `MAX_FRAMES` to a positive number only for a smoke test.

In [ ]:
%pip install -q "ultralytics==8.4.123" pandas pyarrow pillow


In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import time
import traceback
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO, __version__ as ultralytics_version
from ultralytics.engine.model import Model as UltralyticsModel
from ultralytics.models.yolo.detect import DetectionPredictor
from ultralytics.models.yolo.segment import SegmentationPredictor

BATCH_NOTEBOOK_VERSION = '2026-08-20-object-detection-all-videos-v1'
MODEL_NAME = 'yolo26m.pt'
# Faster fixed-vocabulary alternative (lower recall): MODEL_NAME = 'yolo26s.pt'
PIPELINE_VERSION = f'aicv3-object-{Path(MODEL_NAME).stem}-coco-full-video'
MODEL_SCALE = next((scale for scale in 'nsmlx' if f'26{scale}' in MODEL_NAME.lower()), 'm')
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.70
IMAGE_SIZE = 640
MAX_DETECTIONS = 500
END2END = False
AGNOSTIC_NMS_OVERRIDE = None
BATCH_SIZE_OVERRIDE = None
MAX_VIDEOS = None  # Positive integer for a smoke test.
MAX_FRAMES = None  # Positive integer per video for a smoke test.
MAKE_ARCHIVE = False
INPUT_ROOT = Path('/kaggle/input')
BATCH_ROOT = Path('/kaggle/working/05-object-detection-all-videos-output')
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
INFERENCE_PRECISION = 16 if torch.cuda.is_available() else 32

def choose_batch_size() -> int:
    if not torch.cuda.is_available():
        return 1
    vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if MODEL_NAME.lower().startswith('yoloe'):
        if MODEL_SCALE in {'l', 'x'}:
            return 2 if MODEL_SCALE == 'l' and vram_gib >= 24 else 1
        if MODEL_SCALE == 's':
            return 8 if vram_gib >= 24 else 4 if vram_gib >= 16 else 2 if vram_gib >= 12 else 1
        if MODEL_SCALE == 'n':
            return 8 if vram_gib >= 16 else 4 if vram_gib >= 10 else 2
        return 4 if vram_gib >= 24 else 2 if vram_gib >= 16 else 1
    return 16 if vram_gib >= 22 else 12 if vram_gib >= 14 else 8 if vram_gib >= 10 else 4

BATCH_SIZE = BATCH_SIZE_OVERRIDE or choose_batch_size()
AGNOSTIC_NMS = MODEL_NAME.lower().startswith('yoloe') if AGNOSTIC_NMS_OVERRIDE is None else AGNOSTIC_NMS_OVERRIDE
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

if MAX_VIDEOS is not None and int(MAX_VIDEOS) <= 0:
    raise ValueError('MAX_VIDEOS must be positive or None')
if MAX_FRAMES is not None and int(MAX_FRAMES) <= 0:
    raise ValueError('MAX_FRAMES must be positive or None')

KEYFRAME_PATHS = sorted(INPUT_ROOT.rglob('keyframes.parquet'))
if not KEYFRAME_PATHS:
    raise FileNotFoundError('No keyframes.parquet found under /kaggle/input')

def discover_video_manifests(paths: list[Path]):
    grouped, invalid = {}, []
    for path in paths:
        try:
            ids = sorted(pd.read_parquet(path, columns=['video_id'])['video_id'].dropna().astype(str).unique())
        except Exception as exc:
            invalid.append({'manifest': str(path), 'error': f'{type(exc).__name__}: {exc}'})
            continue
        if not ids:
            invalid.append({'manifest': str(path), 'error': 'manifest has no video_id rows'})
        for video_id in ids:
            grouped.setdefault(video_id, []).append(path)
    selected, duplicates = [], {}
    for video_id, candidates in sorted(grouped.items()):
        unique = sorted({str(path) for path in candidates})
        selected.append((video_id, Path(unique[0])))
        if len(unique) > 1:
            duplicates[video_id] = unique
            print(f'WARNING {video_id}: multiple manifests; using {unique[0]}')
    return selected, invalid, duplicates

VIDEO_MANIFESTS, INVALID_MANIFESTS, DUPLICATE_MANIFESTS = discover_video_manifests(KEYFRAME_PATHS)
if MAX_VIDEOS is not None:
    VIDEO_MANIFESTS = VIDEO_MANIFESTS[:int(MAX_VIDEOS)]
if not VIDEO_MANIFESTS:
    raise RuntimeError('No video manifests with usable video_id values.')

def success_matches_configuration(path: Path, source_manifest: Path) -> bool:
    if not path.exists():
        return False
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    return (payload.get('batch_notebook_version') == BATCH_NOTEBOOK_VERSION
            and payload.get('pipeline_version') == PIPELINE_VERSION
            and Path(str(payload.get('source_manifest', ''))).as_posix() == Path(source_manifest).as_posix()
            and payload.get('model') == MODEL_NAME
            and payload.get('max_frames') == MAX_FRAMES
            and payload.get('archive_enabled') == MAKE_ARCHIVE)

print('Batch notebook version:', BATCH_NOTEBOOK_VERSION)
print('Keyframe manifests found:', len(KEYFRAME_PATHS))
print('Videos discovered:', len(VIDEO_MANIFESTS))
print('Model:', MODEL_NAME, '| image size:', IMAGE_SIZE, '| batch size:', BATCH_SIZE)
print('Ultralytics:', ultralytics_version)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Load once and reuse for every video.
model = YOLO(MODEL_NAME)
print('YOLO model ready:', MODEL_NAME, '| task:', getattr(model, 'task', None))


In [ ]:
def resolve_frame_path(value: str, input_root: Path, frame_root: Path) -> Path:
    for candidate in (input_root / value, frame_root.parent / value):
        if candidate.exists():
            return candidate
    matches = list(input_root.rglob(Path(value).name))
    if matches:
        return matches[0]
    raise FileNotFoundError(value)

def canonical_path(value: str | Path) -> str:
    return str(Path(value).resolve())

class BoxesOnlySegmentationPredictor(SegmentationPredictor):
    '''Skip mask construction because this stage stores bounding boxes only.'''
    def postprocess(self, preds, img, orig_imgs, **kwargs):
        return DetectionPredictor.postprocess(self, preds[0], img, orig_imgs, **kwargs)
    def construct_results(self, preds, img, orig_imgs, protos=None):
        return DetectionPredictor.construct_results(self, preds, img, orig_imgs)
    def construct_result(self, pred, img, orig_img, img_path, proto=None):
        return DetectionPredictor.construct_result(self, pred, img, orig_img, img_path)

def as_float(value) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float('nan')

def cuda_sync() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def process_video(keyframes_path: Path, video_id: str, output_root: Path) -> dict:
    output_root.mkdir(parents=True, exist_ok=True)
    input_root = keyframes_path.parent
    frame_roots = sorted(input_root.rglob('frames'))
    frame_root = frame_roots[0] if frame_roots else input_root / 'frames'
    keyframes_df = pd.read_parquet(keyframes_path)
    required = {'keyframe_id', 'video_id', 'frame_idx', 'pts_time', 'frame_path'}
    missing = required - set(keyframes_df.columns)
    if missing:
        raise ValueError(f'Missing keyframe columns: {sorted(missing)}')
    keyframes_df = keyframes_df[keyframes_df.video_id.astype(str) == str(video_id)].sort_values('frame_idx').reset_index(drop=True)
    if keyframes_df.empty:
        raise ValueError(f'No keyframes found for {video_id}')
    if MAX_FRAMES is not None and int(MAX_FRAMES) < len(keyframes_df):
        indices = np.linspace(0, len(keyframes_df) - 1, num=int(MAX_FRAMES), dtype=int)
        keyframes_df = keyframes_df.iloc[np.unique(indices)].reset_index(drop=True)

    detection_columns = ['detection_id', 'model', 'weights', 'keyframe_id', 'video_id', 'frame_idx', 'pts_time', 'frame_path', 'class_id', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'bbox_width', 'bbox_height', 'bbox_area_ratio', 'width', 'height', 'x1_norm', 'y1_norm', 'x2_norm', 'y2_norm', 'bbox_norm']
    timing_columns = ['model', 'weights', 'keyframe_id', 'video_id', 'frame_idx', 'pts_time', 'preprocess_ms', 'inference_ms', 'postprocess_ms', 'reported_total_ms', 'detections']
    paths, metadata_by_path, resolved_by_keyframe, errors = [], defaultdict(deque), {}, []
    for row in keyframes_df.itertuples(index=False):
        try:
            path = resolve_frame_path(str(row.frame_path), input_root, frame_root)
            key = canonical_path(path)
            paths.append(key)
            metadata_by_path[key].append(row)
            resolved_by_keyframe[str(row.keyframe_id)] = key
        except Exception as exc:
            errors.append({'keyframe_id': str(row.keyframe_id), 'frame_path': str(row.frame_path), 'error': str(exc)})
    if not paths:
        raise RuntimeError(f'No readable keyframe images were found for {video_id}.')
    (output_root / 'object-errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    source_list = output_root / 'resolved-frames.txt'
    source_list.write_text('\n'.join(paths), encoding='utf-8')
    print('Keyframes:', len(keyframes_df), '| resolved:', len(paths), '| path errors:', len(errors))

    records, timings, detection_counter, result_count = [], [], 0, 0
    metadata_queues = {path: deque(rows) for path, rows in metadata_by_path.items()}
    predict_kwargs = {'source': str(source_list), 'stream': True, 'batch': BATCH_SIZE, 'conf': CONFIDENCE_THRESHOLD, 'iou': IOU_THRESHOLD, 'imgsz': IMAGE_SIZE, 'device': DEVICE, 'quantize': INFERENCE_PRECISION, 'max_det': MAX_DETECTIONS, 'end2end': END2END, 'agnostic_nms': AGNOSTIC_NMS, 'rect': True, 'save': False, 'verbose': False}
    if getattr(model, 'task', None) == 'segment':
        result_stream = UltralyticsModel.predict(model, predictor=BoxesOnlySegmentationPredictor, **predict_kwargs)
    else:
        result_stream = model.predict(**predict_kwargs)

    cuda_sync()
    run_start = time.perf_counter()
    for result_count, result in enumerate(result_stream, start=1):
        row_queue = metadata_queues.get(canonical_path(result.path))
        if not row_queue:
            raise RuntimeError(f'No keyframe metadata for inference result: {result.path}')
        row = row_queue.popleft()
        speed = result.speed or {}
        pre, infer, post = as_float(speed.get('preprocess')), as_float(speed.get('inference')), as_float(speed.get('postprocess'))
        boxes = result.boxes
        count = 0 if boxes is None else int(len(boxes))
        timings.append({'model': MODEL_NAME, 'weights': MODEL_NAME, 'keyframe_id': str(row.keyframe_id), 'video_id': str(row.video_id), 'frame_idx': int(row.frame_idx), 'pts_time': float(row.pts_time), 'preprocess_ms': pre, 'inference_ms': infer, 'postprocess_ms': post, 'reported_total_ms': pre + infer + post, 'detections': count})
        if boxes is None or len(boxes) == 0:
            continue
        xyxy = boxes.xyxy.detach().cpu().numpy()
        confs = boxes.conf.detach().cpu().numpy()
        classes = boxes.cls.detach().cpu().numpy().astype(int)
        names, shape = result.names, getattr(result, 'orig_shape', None)
        if shape is None or len(shape) < 2:
            raise RuntimeError(f'Missing original image shape for {row.keyframe_id}')
        height, width = int(shape[0]), int(shape[1])
        for box, confidence, class_id in zip(xyxy, confs, classes):
            detection_counter += 1
            x1, y1, x2, y2 = [float(value) for value in box]
            bbox_width, bbox_height = max(0.0, x2 - x1), max(0.0, y2 - y1)
            class_name = str(names.get(int(class_id), class_id) if isinstance(names, dict) else names[int(class_id)])
            records.append({'detection_id': f'{MODEL_NAME}:{row.keyframe_id}:det:{detection_counter:06d}', 'model': MODEL_NAME, 'weights': MODEL_NAME, 'keyframe_id': str(row.keyframe_id), 'video_id': str(row.video_id), 'frame_idx': int(row.frame_idx), 'pts_time': float(row.pts_time), 'frame_path': str(row.frame_path), 'class_id': int(class_id), 'class_name': class_name, 'confidence': float(confidence), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'bbox_width': bbox_width, 'bbox_height': bbox_height, 'bbox_area_ratio': bbox_width * bbox_height / float(width * height), 'width': width, 'height': height, 'x1_norm': x1 / width, 'y1_norm': y1 / height, 'x2_norm': x2 / width, 'y2_norm': y2 / height, 'bbox_norm': [x1 / width, y1 / height, x2 / width, y2 / height]})

    cuda_sync()
    run_wall_sec = time.perf_counter() - run_start
    for row_queue in metadata_queues.values():
        for row in row_queue:
            errors.append({'keyframe_id': str(row.keyframe_id), 'frame_path': str(row.frame_path), 'error': 'Ultralytics did not return a result; the image may be unreadable.'})
    (output_root / 'object-errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    objects_df = pd.DataFrame(records, columns=detection_columns)
    timings_df = pd.DataFrame(timings, columns=timing_columns)
    objects_df.to_parquet(output_root / 'objects.parquet', index=False)
    objects_df.to_json(output_root / 'objects.jsonl', orient='records', lines=True, force_ascii=False)
    objects_df.to_csv(output_root / 'objects.csv', index=False)
    timings_df.to_csv(output_root / 'frame-timings.csv', index=False)
    if objects_df.empty:
        class_summary_df = pd.DataFrame(columns=['class_name', 'detections', 'frames', 'avg_confidence'])
    else:
        class_summary_df = objects_df.groupby('class_name', as_index=False).agg(detections=('detection_id', 'count'), frames=('keyframe_id', 'nunique'), avg_confidence=('confidence', 'mean')).sort_values(['detections', 'avg_confidence'], ascending=False)
    class_summary_df.to_csv(output_root / 'detected-classes.csv', index=False)
    archive = None
    if MAKE_ARCHIVE:
        archive = shutil.make_archive(str(BATCH_ROOT / f'05-object-detection-{video_id}'), 'zip', root_dir=output_root)
        print('Download:', archive)
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None
    summary = {'stage': 'object-detection', 'batch_notebook_version': BATCH_NOTEBOOK_VERSION, 'pipeline_version': PIPELINE_VERSION, 'source_manifest': str(keyframes_path), 'video_id': str(video_id), 'model': MODEL_NAME, 'ultralytics_version': ultralytics_version, 'vocabulary_mode': 'prompt-free-open-vocabulary' if MODEL_NAME.lower().startswith('yoloe') else 'coco-fixed-vocabulary', 'keyframes': int(len(keyframes_df)), 'frames_resolved': int(len(paths)), 'frames_read': int(result_count), 'frame_errors': int(len(errors)), 'detections': int(len(objects_df)), 'unique_frames_with_objects': int((timings_df.detections > 0).sum()), 'confidence_threshold': CONFIDENCE_THRESHOLD, 'iou_threshold': IOU_THRESHOLD, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'end2end': END2END, 'precision': f'FP{INFERENCE_PRECISION}', 'boxes_only': getattr(model, 'task', None) == 'segment', 'disk_streaming': True, 'measured_wall_sec': run_wall_sec, 'measured_throughput_fps': result_count / run_wall_sec if run_wall_sec > 0 else None, 'peak_gpu_memory_mb': peak_gpu_memory_mb, 'max_frames': MAX_FRAMES, 'archive_enabled': MAKE_ARCHIVE, 'archive': archive}
    (output_root / 'object-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    display(class_summary_df.head(20))
    return summary


In [ ]:
success_count = skipped_count = failure_count = 0
failures = []
for video_number, (video_id, keyframes_path) in enumerate(VIDEO_MANIFESTS, start=1):
    output_root = BATCH_ROOT / video_id
    success_path, error_path = output_root / '_SUCCESS.json', output_root / '_ERROR.txt'
    if success_matches_configuration(success_path, keyframes_path):
        success_count += 1
        skipped_count += 1
        print(f'[{video_number}/{len(VIDEO_MANIFESTS)}] Skip completed {video_id}')
        continue
    print(f'\n[{video_number}/{len(VIDEO_MANIFESTS)}] Processing {video_id}')
    print('Manifest:', keyframes_path)
    success_path.unlink(missing_ok=True)
    try:
        summary = process_video(keyframes_path, video_id, output_root)
        success_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        error_path.unlink(missing_ok=True)
        success_count += 1
    except Exception as exc:
        failure_count += 1
        error_path.parent.mkdir(parents=True, exist_ok=True)
        error_path.write_text(traceback.format_exc(), encoding='utf-8')
        failures.append({'video_id': str(video_id), 'manifest': str(keyframes_path), 'error': f'{type(exc).__name__}: {exc}'})
        print(f'ERROR {video_id}: {type(exc).__name__}: {exc}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
batch_summary = {'stage': 'object-detection', 'batch_notebook_version': BATCH_NOTEBOOK_VERSION, 'pipeline_version': PIPELINE_VERSION, 'manifests_found': len(KEYFRAME_PATHS), 'videos_discovered': len(VIDEO_MANIFESTS), 'completed': success_count, 'skipped_existing': skipped_count, 'failed': failure_count, 'invalid_manifests': INVALID_MANIFESTS, 'duplicate_manifests': DUPLICATE_MANIFESTS, 'failures': failures}
(BATCH_ROOT / 'batch-summary.json').write_text(json.dumps(batch_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))
if failure_count:
    print('Some videos failed. Inspect each _ERROR.txt; rerun to retry videos without _SUCCESS.json.')
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Output

Each video is saved under `/kaggle/working/05-object-detection-all-videos-output/<VIDEO_ID>/`, including `objects.parquet`, `objects.jsonl`, `objects.csv`, `frame-timings.csv`, `detected-classes.csv`, `object-summary.json`, and `_SUCCESS.json`. The batch-level result is `batch-summary.json`.